In [10]:
import pandas as pd
import math
import numpy as np
import matplotlib.pyplot as plt
import random as rd


In [11]:
#Import set of generated answers for this set of users
df_data_2 = pd.read_csv("generated_results/data_sample_2_users_2_groups.csv")
df_data_2

df_data_5 = pd.read_csv("generated_results/data_sample_5_users_2_groups.csv")
df_data_5

df_data_10 = pd.read_csv("generated_results/data_sample_10_users_2_groups.csv")
df_data_10

,User_ID,Group,ActiveEntity,SufficientInformation,SpecificSufficientInformation - have,SpecificSufficientInformation - know,SpecificSufficientInformation - are
0,0,1,Processes,Smtg they have,Certificate,PIN,Fingerprint
1,1,2,Human User,smtg they know and smtg they are,Certificate,Text-password,Iris
2,2,1,Human User,Smtg they know and smtg they have,Token,Text-password,Retina
3,3,1,Both,Smtg they know,Certificate,Text-password,Iris
4,4,1,Human User,smtg they know and smtg they are,Certificate,PIN,Fingerprint
5,5,1,Processes,smtg they know and smtg they are,Token,PIN,Retina
6,6,2,Both,Smtg they are,Assertion,Text-password,Voice
7,7,1,Processes,Smtg they are,Assertion,Pattern password,Retina
8,8,2,Processes,Smtg they are,Token,Pattern password,Voice
9,9,2,Processes,Smtg they know and smtg they have,Token,Passphrase,Iris


In [12]:
#Methods applied to all of these parameters and the option we get at the end

# Majority
# df : dataframe we have with the user/group and parameters we are concerned about
# maj_profile : the profiles we apply majority to
# We could have a separate majority mechanism where we want majority in each of the profiles provided
def majority(df,maj_profile) :
    #create df storing results for each parameter with the option if there was a majority
    df_results = dict.fromkeys(df.columns.copy()[2:])
    df_group = df.query(maj_profile)
    nb_user = df_group["User_ID"].count()
    for parameter in df_results.keys() :
        
       # Retrieve number of votes for each option
        votes = df_group[parameter].value_counts().to_dict()
        print(votes)
        temp_arr = []
        
        for key,val in votes.items() :
            if val >= math.ceil(nb_user) / 2 :
                temp_arr.append(key)
        if (len(temp_arr) == 1) :
            # print(temp_arr)
            df_results[parameter] = temp_arr[0]
            
    print(df_results)
    
# Unanimity
# df : dataframe we have with the user/group and parameters we are concerned about
# maj_profile : the profiles we apply majority to
def unanimity(df,maj_profile) :
    df_results = dict.fromkeys(df.columns.copy()[2:])
    df_group = df.query(maj_profile)
    nb_user = df_group["User_ID"].count()
    for parameter in df_results.keys() :
        
       # Retrieve number of votes for each option
        votes = df_group[parameter].value_counts().to_dict()
        for key,val in votes.items() :
            if val == nb_user :
                df_results[parameter] = key
            
    print(df_results)
    
# Average
# df : dataframe we have with the user/group and parameters we are concerned about
# maj_profile : the profiles we apply majority to
def average(df,maj_profile) :
    df_results = dict.fromkeys(df.columns.copy()[2:])
    df_results_rounded = dict.fromkeys(df.columns.copy()[2:])
    df_group = df.query(maj_profile)
    nb_user = df_group["User_ID"].count()
    
    for parameter in df_results.keys() :
        df_results[parameter] = df_group[parameter].mean()
        df_results_rounded[parameter] = math.ceil(df_group[parameter].mean())
    print(df_results)
    print(df_results_rounded)

# Multiplicative in the votes made 
# df : dataframe we have with the user/group and parameters we are concerned about
# mult : dictionary with the key corresponding to the profile and value corresponding to the multiplicative power
def multiplicative(df,mult) :
    df_votes = pd.DataFrame(columns=df.columns.copy()[1:],index=df["Group"].unique())
    df_votes["Group"]= df["Group"].unique().copy()
    df_results = dict.fromkeys(df.columns.copy()[2:])
    for group in df["Group"].unique() :
        df_group = df.query(f"Group == {group}")
        # print(df_group)
        for parameter in df_results.keys() :
       # Retrieve number of votes for each option
            votes = df_group[parameter].value_counts().to_dict()
            # update the votes with the multiplicative power of the group
            votes.update({k: v* mult.get(group) for k,v in votes.items()})
            # votes = votes * mult.get(group)
            df_votes.at[group,parameter] = votes
    return df_votes
        
# majority(df_data_2,"Group == 1 | Group == 2") #df_data["Group"].unique()
# majority(df_data_2,"Group == 1")
# unanimity(df_data_2,"Group == 1 | Group == 2") 



# average(df_data,"Group == 1 | Group == 2")
# multiplicative(df_data,"Group == 1 | Group == 2",[])

In [ ]:
# Represent votes for each parameter with its answers
def plot_bar_answers(df_data,threshold) :
    for param in df_data.columns[2:] :
        dct = df_data[param].value_counts().to_dict()
        fig,ax = plt.subplots()
        ax.bar(x=list(dct.keys()),height=dct.values(),width=0.5)
        # ax.set(xlim=len(df_data_2))
        len_df =  len(df_data)
        # xlim=(0,len_df), xticks=np.arange(1, len_df),
        ax.set(
        ylim=(0, len_df), yticks=np.arange(1, len_df))
        ax.set_title(f"PARAM - {param} | {len(df_data)} USERS")
        
        #THRESHOLD
        line = len(df_data.index)*threshold/100
        plt.axhline(line,color="red",linestyle="--")
        plt.text(line /2,line +.05,'threshold',rotation=0,color="red")
        
        plt.show()
    # ax = df_data_2.plot(x=list(dct.keys()),y=dct.values())
    # print(param,dct.keys(),dct.values())
    # ax.set_xlim(0,2)
    # ax.figure.legend(param)
    # ax.grid()
    # ax.set_title(f"Votes of Parameter{param}")
    # plt.xticks(range(-1,len(df_data_2["User_ID"])+1))
    # plt.show()
  
def plot_bar_answers_bis(df_data,threshold,votes) :
    votes_qu = votes.loc[:,votes.columns != "Group"]
    display(votes_qu)
    for name,val in votes_qu.items() : 
         
        val_cb = {}
        for idx in range(1,len(val)+1) :
            
        #    for x in val[idx].keys() :
        #     if x in val_cb :
        #         print(x)
        #         val_cb = val_cb[idx] + val[idx]
        #     else :
            
            for key in val[idx].keys() :
                
                if key in val_cb :
                    print(f"hello {key}")
                    val_cb[key] = val_cb[key] +val[idx].get(key)
                # print(val_cb.keys(),val[idx].keys())
                else :
                    val_cb[key] = val[idx].get(key)        # print(val_cb)
        print(val_cb)
        fig,ax = plt.subplots()
        ax.bar(x=list(val_cb.keys()),height=val_cb.values(),width=0.5)
        # ax.set(xlim=len(df_data_2))
        len_df =  len(df_data)
        # xlim=(0,len_df), xticks=np.arange(1, len_df),
        ax.set(
        ylim=(0, len_df), yticks=np.arange(1, len_df))
        ax.set_title(f"PARAM - {name} | {len(df_data)} USERS")
        
        #THRESHOLD
        line = len(df_data.index)*threshold/100
        plt.axhline(line,color="red",linestyle="--")
        plt.text(len(val_cb.keys())/2,line +.05,'threshold',rotation=0,color="red")
        
        plt.show() 
        
# plot_bar_answers(df_data_2,90)
# plot_bar_answers(df_data_5,40)
# plot_bar_answers(df_data_10,50)

df_votes = multiplicative(df_data_5,{1 : 1.5, 2 : 2.5})
plot_bar_answers(df_data_5,40)
plot_bar_answers_bis(df_data_5,60,df_votes)
    

IndentationError: unexpected indent (4038799782.py, line 65)

In [ ]:
# As we can see some decisions stay unresolved -> try with multiplicity method
def multiplicate(df_data,dict_group_mult) :
    for index,row in df_data.iterrows() :
        val_grp_user = dict_group_mult.get(row["Group"])
        row[:2] = row[:2] * val_grp_user
        print(df_data)

dict_grp_mul = { 1 : 1.5, 2 : 3}
# print(df_data_2)
multiplicate(df_data_2,dict_grp_mul)

   User_ID  Group ActiveEntity              SufficientInformation  \
0        0      1         Both                     Smtg they have   
1        1      2   Human User  Smtg they know and smtg they have   

  SpecificSufficientInformation - have SpecificSufficientInformation - know  \
0                          Certificate                        Text-password   
1                                Token                                  PIN   

  SpecificSufficientInformation - are  
0                  Facial Recognition  
1                               Voice  
   User_ID  Group ActiveEntity              SufficientInformation  \
0        0      1         Both                     Smtg they have   
1        1      2   Human User  Smtg they know and smtg they have   

  SpecificSufficientInformation - have SpecificSufficientInformation - know  \
0                          Certificate                        Text-password   
1                                Token                              

In [ ]:
# Takes as parameters : the parameter (ex:SADD1P1) the profile/group we focus on 
# Return true or false whether we have majority or not
def majority(param,df_profile) : 
    return param > math.ceil(len(df_profile)) / 2

# Return true or false whether we have unanimity or not 
def unanimity(param,df_profile) :
    return param == len(df_profile)

# average of the answers given (but answers are not ordered here) for a parameter
# Could be related to the average of distance we have between the options selected
def average(param,df_profile) :
    return 

# Ranking of alternatives (doing this on all the choices would be too much)
def bordaCount(param,df_profile) :
    return 0

# Setting up multiplication related to profiles
def multiplicative(param,df_profile) :
    return 0

def are_all_options_selected(result_table) :
    for val in result_table.values() :
        if val == None :
            return False
    return True

def get_parameters_left(result_table) :
    parameters_left = []
    for key,val in result_table.items() :
        if val is None : 
            parameters_left.append(key)
    return parameters_left
  
def show_options_selected(result_table) :
    for key,val in result_table.items() :
        if (val is None) :
            print(f"Parameter {key} doesn't have a selected option yet")
        else :
            print(f"Parameter {key}, O{val} was selected")

def check_criterion(result_table,df_users,group_query,condition) :
    if not (are_all_options_selected(result_table)) :
        parameters_name = get_parameters_left(result_table)
        
        df_group = df_users.query(group_query)
        #For each parameter 
        for parameter_name in parameters_name :
                            
            #Get the values for this parameter
            parameter_values = df_group[parameter_name].value_counts()
            for index,param_value in parameter_values.items() :
                
                #Check for majority
                if condition == "majority" :
                    if majority(param_value,df_group):
                        result_table.update({parameter_name : index })
                    
                #Check for unanimity
                elif condition == "unanimity" :
                    if unanimity(param_value,df_group):
                        result_table.update({parameter_name : index })

def remove_linked_parameters(result_table,linked_parameters) :
    keys_to_remove = []
    for lnk_param in linked_parameters :
        for key,val in result_table.items() :
            if lnk_param[0] == key and lnk_param[1] != val :
                keys_to_remove.append(lnk_param[2])
    for key in keys_to_remove :
        print(f"Parameter {key} was removed as its linked parameter option was not selected.")
        result_table.pop(key)

result_table = dict.fromkeys(df_data.columns.copy()[2:])

linked_parameters = [["SADD1P3",3,"SADD1P5"]]

# First set of Resolution Strategies

# Unanimity for Expert and Advanced
check_criterion(result_table,df_data," Group == 1 | Group == 2","unanimity")

# If we still have options to resolve
if not (are_all_options_selected(result_table)) :
    # We apply the second strategy Unanimity for Expert
    check_criterion(result_table,df_data,"Group == 2","unanimity")

show_options_selected(result_table)


NameError: name 'df_data' is not defined

In [ ]:
def get_least_vote(threshold,df_data,val) :
    size = df_data[val].unique().size
    serie_opt = df_data[val].value_counts().reindex(range(1,size+1), fill_value=0)
    limit = serie_opt.sum()*threshold
    opt_to_remove = []
    for index, value in serie_opt.items() :
        if (value < limit) :
            opt_to_remove.append(index)
    return opt_to_remove

def remove_least_voted_options(df_data,result_table,threshold) :
    params = get_parameters_left(result_table)
    param_options_to_remove = {}
    for val in params :
        param_options_to_remove.update({val : get_least_vote(threshold,df_data,val)})
        
    return param_options_to_remove

# Reduction of parameters with least votes according to the criterias with a threshold of 15%
p_opt_to_rm = remove_least_voted_options(df_data,result_table,0.15)

print(p_opt_to_rm)


{'P2': [], 'P5': [3], 'P6': [], 'P7': [2]}


In [ ]:
# Loading the second questionnaire result
df_data_2 = pd.read_csv("SADD_Options_Selected_2.csv",sep=",")
df_data_2.sort_values(by="Group")

,User_ID,Group,SADD1P2,SADD1P3,SADD1P5
5,9,1,1,3,4.0
6,10,1,1,3,2.0
7,7,1,2,3,2.0
8,8,1,2,3,1.0
9,6,1,2,2,NaN
0,14,2,1,2,NaN
1,15,2,1,2,NaN
2,11,2,1,2,NaN
3,12,2,1,2,NaN
4,13,2,2,3,2.0


In [ ]:
# Majority for Expert and Advanced
check_criteria(result_table,df_data_2," Group == 1 | Group == 2","majority")
print(result_table)
if not (are_all_options_selected(result_table)) :
    # Majority for Expert
    check_criteria(result_table,df_data,"Group == 2","majority")
remove_linked_parameters(result_table,linked_parameters)
show_options_selected(result_table)

NameError: name 'check_criteria' is not defined